In [5]:
import pandas as pd

# 1. Carregar os arquivos CSV
print("Carregando os CSVs...")
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# 2. Separa a variável alvo (Target) do dataset de treino
y_train = train_df['winner']

# Guardamos os nomes das seleções do teste para usar no CSV final
times_teste = test_df['team_name']

# 3. Removemos as colunas de texto que não são variáveis preditivas (Identificadores e Target)
# Não podemos enviar texto puro (strings) para o modelo do sklearn
features_train = train_df.drop(columns=['winner', 'team_name', 'country_code'])
features_test = test_df.drop(columns=['team_name', 'country_code'])

# 4. Concatenamos o Treino e Teste temporariamente
# Porque se o treino tiver uma confederação que não existe no teste, 
# o One-Hot Encoding vai criar um número diferente de colunas e o modelo vai se arrebentar.
# Juntamos os dois para garantir que o pd.get_dummies cria as colunas exatas para ambos.
df_combined = pd.concat([features_train, features_test])

# 5. One-Hot Encoding: Transformar a coluna 'confederation' (texto) em colunas binárias numéricas (0 ou 1)
df_combined = pd.get_dummies(df_combined, columns=['confederation'])

# 6. Separar novamente em Treino  e Teste 
# Usamos o tamanho original do train_df para fazer o corte (slicing) no exato ponto certo
X_train = df_combined.iloc[:len(train_df)].copy()
X_test_final = df_combined.iloc[len(train_df):].copy()

# 7. Verificação final de sanidade (Opcional, mas recomendado para garantir que não há dados nulos)
X_train.fillna(0, inplace=True)
X_test_final.fillna(0, inplace=True)

print("Pré-processamento concluído com sucesso!\n")
print(f"\nFormato final dos dados de treino: {X_train.shape}")
print(f"\nFormato final dos dados de teste: {X_test_final.shape}")

Carregando os CSVs...
Pré-processamento concluído com sucesso!


Formato final dos dados de treino: (1000, 28)

Formato final dos dados de teste: (250, 28)


## Task 3 - Previsões oficiais e Feature Importance

Esta etapa aplica o modelo treinado no `test.csv`, gera o ranking de probabilidades e lista as variáveis que mais influenciaram a decisão do modelo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# A Task 3 depende do modelo treinado pela Task 2.
# Se alguém já tiver criado um modelo no notebook, reaproveitamos. Caso contrário,
# treinamos um RandomForest básico para conseguir gerar a entrega oficial.
if 'modelo_treinado' in globals():
    modelo = modelo_treinado
elif 'modelo' not in globals():
    modelo = RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight='balanced'
    )
    modelo.fit(X_train, y_train)

print("Modelo pronto para gerar as previsões oficiais.")

In [ ]:
from pathlib import Path

outputs_dir = Path('../outputs')
outputs_dir.mkdir(exist_ok=True)

# predict_proba retorna a probabilidade de cada classe.
# Como winner=1 representa campeão/vencedor, pegamos a coluna da classe 1.
classes_modelo = list(modelo.classes_)
classe_vencedor = 1 if 1 in classes_modelo else classes_modelo[-1]
indice_classe_vencedor = classes_modelo.index(classe_vencedor)
probabilidades_vitoria = modelo.predict_proba(X_test_final)[:, indice_classe_vencedor]

previsoes_test = pd.DataFrame({
    'team_name': times_teste,
    'country_code': test_df['country_code'],
    'confederation': test_df['confederation'],
    'probabilidade_campeao': probabilidades_vitoria,
    'probabilidade_campeao_pct': (probabilidades_vitoria * 100).round(2)
})

previsoes_detalhadas_path = outputs_dir / 'previsoes_detalhadas_test.csv'
previsoes_test.to_csv(previsoes_detalhadas_path, index=False, encoding='utf-8-sig')

# O test.csv possui seleções repetidas. Para o ranking oficial, agregamos por seleção
# usando a probabilidade média prevista pelo modelo.
ranking_probabilidades = (
    previsoes_test
    .groupby(['team_name', 'country_code', 'confederation'], as_index=False)
    .agg(
        probabilidade_campeao=('probabilidade_campeao', 'mean'),
        qtd_amostras=('probabilidade_campeao', 'size')
    )
)
ranking_probabilidades['probabilidade_campeao_pct'] = (
    ranking_probabilidades['probabilidade_campeao'] * 100
).round(2)

ranking_probabilidades = (
    ranking_probabilidades
    .sort_values(['probabilidade_campeao', 'team_name'], ascending=[False, True])
    .reset_index(drop=True)
)
ranking_probabilidades.insert(0, 'posicao', ranking_probabilidades.index + 1)

ranking_path = outputs_dir / 'ranking_probabilidades.csv'
ranking_probabilidades.to_csv(ranking_path, index=False, encoding='utf-8-sig')

# Feature Importance: mostra quais colunas tiveram mais peso no modelo.
if hasattr(modelo, 'feature_importances_'):
    importancias = modelo.feature_importances_
elif hasattr(modelo, 'coef_'):
    importancias = abs(modelo.coef_).ravel()
else:
    raise AttributeError('O modelo usado não possui feature_importances_ nem coef_.')

feature_importance = pd.DataFrame({
    'variavel': X_train.columns,
    'importancia': importancias
})

feature_importance = (
    feature_importance
    .sort_values('importancia', ascending=False)
    .reset_index(drop=True)
)

feature_importance_path = outputs_dir / 'feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False, encoding='utf-8-sig')

print(f"Previsões detalhadas salvas em: {previsoes_detalhadas_path}")
print(f"Ranking salvo em: {ranking_path}")
print(f"Feature importance salvo em: {feature_importance_path}")

print("\nTop 10 seleções por probabilidade:")
display(ranking_probabilidades.head(10))

print("\nTop 10 variáveis mais importantes:")
display(feature_importance.head(10))